# Verificación: carga de tablas MIMIC-IV-ED

Carga directa desde el path configurado en `.env`. Se verifica que las tablas `edstays` y `triage` son accesibles y tienen la estructura esperada, incluyendo distribución de disposiciones y tasas de valores faltantes en las constantes vitales de triaje.

In [1]:
import os
import logging
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

logging.basicConfig(level=logging.INFO, format="%(levelname)s — %(message)s", force=True)

# Carga del .env desde la ruta del notebook o desde la raíz del proyecto
load_dotenv(dotenv_path=Path("../../.env"), override=True)
if not os.getenv("MIMIC_IV_ED_PATH"):
    load_dotenv(dotenv_path=Path(".env"), override=True)

DATA = Path(os.getenv("MIMIC_IV_ED_PATH", ""))
print(f"DATA path : {DATA}")
print(f"Existe    : {DATA.exists()}")

DATA path : C:\Users\cuent\Documents\UAX\TFM\TFM\CodigoGit\data
Existe    : True


In [2]:
# Carga de edstays con parseo de fechas
df_stays = pd.read_csv(DATA / "edstays.csv", low_memory=False,
                       parse_dates=["intime", "outtime"])
print("edstays:", df_stays.shape)
print(df_stays.columns.tolist())
df_stays.head(3)

edstays: (425087, 9)
['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime', 'gender', 'race', 'arrival_transport', 'disposition']


,subject_id,hadm_id,stay_id,intime,outtime,gender,race,arrival_transport,disposition
0,10000032,22595853.0,33258284,2180-05-06 19:17:00,2180-05-06 23:30:00,F,WHITE,AMBULANCE,ADMITTED
1,10000032,22841357.0,38112554,2180-06-26 15:54:00,2180-06-26 21:31:00,F,WHITE,AMBULANCE,ADMITTED
2,10000032,25742920.0,35968195,2180-08-05 20:58:00,2180-08-06 01:44:00,F,WHITE,AMBULANCE,ADMITTED


In [3]:
# Carga de triage
df_triage = pd.read_csv(DATA / "triage.csv", low_memory=False)
print("triage:", df_triage.shape)
print(df_triage.columns.tolist())
df_triage.head(3)

triage: (425087, 11)
['subject_id', 'stay_id', 'temperature', 'heartrate', 'resprate', 'o2sat', 'sbp', 'dbp', 'pain', 'acuity', 'chiefcomplaint']


,subject_id,stay_id,temperature,heartrate,resprate,o2sat,sbp,dbp,pain,acuity,chiefcomplaint
0,10000032,32952584,97.8,87.0,14.0,97.0,71.0,43.0,7,2.0,Hypotension
1,10000032,33258284,98.4,70.0,16.0,97.0,106.0,63.0,0,3.0,"Abd pain, Abdominal distention"
2,10000032,35968195,99.4,105.0,18.0,96.0,106.0,57.0,10,3.0,"n/v/d, Abd pain"


In [4]:
# Distribución de la disposición final del episodio
print(df_stays["disposition"].value_counts())

disposition
HOME                           241632
ADMITTED                       158010
TRANSFER                         7025
LEFT WITHOUT BEING SEEN          6155
ELOPED                           5710
OTHER                            4297
LEFT AGAINST MEDICAL ADVICE      1881
EXPIRED                           377
Name: count, dtype: int64


In [5]:
# Porcentaje de valores faltantes en las variables de triage
miss_pct = (df_triage.isna().sum() / len(df_triage) * 100).round(2)
print(miss_pct[miss_pct > 0].to_string())

temperature       5.51
heartrate         4.02
resprate          4.79
o2sat             4.85
sbp               4.30
dbp               4.49
pain              3.04
acuity            1.64
chiefcomplaint    0.01
